# **Data and AI in Economics: CEO Turnover and Executive Pay Dispersion in Europe: Evidence from Leadership Changes**

# **Table of Contents**

## [**1. Team**](#1-team)

## [**2. Research Framework**](#2-research-framework)

- [2.1 Research Title, Research Question and Motivation](#21-research-title-research-question-and-motivation)

- [2.2 Datasets and Key Variables](#22-datasets-and-key-variables)

  - [2.2.1 Sources](#221-sources)

  - [2.2.2 Key Variables](#222-key-variables)

  - [2.2.3 Dataset Preview](#223-dataset-preview)

- [2.3 Analytic Approach](#23-analytic-approach)

  - [2.3.1 Causal Inference](#231-causal-inference)

  - [2.3.2 Supervised Learning](#232-supervised-learning)

  - [2.3.3 Unsupervised Learning](#233-unsupervised-learning)

- [2.4 Evaluation Metric](#24-evaluation-metric)

- [2.5 Research Work Plan](#25-research-work-plan)

## [**3. Setup**](#3-setup)

- [3.1 Imports and Path Configuration](#31-imports-and-path-configuration)

- [3.2 Helper Functions](#32-helper-functions)

## [**3. Data Understanding (Exploratory Data Analysis)**](#3-data-understanding-exploratory-data-analysis)

- [3.1 Data Inspection and Descriptive Statistics](#31-data-inspection-and-descriptive-statistics)

- [3.2 Data Visualization](#32-data-visualization)

## [**4. Data Preprocessing**](#4-data-preprocessing)

- [4.1 Data Cleaning](#41-data-cleaning)

  - [4.1.1 Missing Values](#411-missing-values)

  - [4.1.2 Duplicated Values](#412-duplicated-values)

  - [4.1.3 Outliers](#413-outliers)

- [4.2 Feature Selection](#42-feature-selection)

## [**5. Causal Inference**](#5-causal-inference)

## [**6. Supervised Learning**](#6-supervised-learning)

## [**7. Unsupervised Learning**](#7-unsupervised-learning)

## [**8. Results and Conclusion**](#8-results-and-conclusion)


---

## **1. Team**

| Role | Name | Student ID |
|------|------|------------|
| Lead |Achmad Rizky Akbar| |
| Member | Kajetan Zduńczyk| |


---

## **2. Research Framework**

### **2.1 Research Title, Research Question and Motivation**

**Research Title:** *CEO Turnover and Executive Pay Dispersion in Europe: Evidence from Leadership Changes*

**Research Question:** *Does CEO turnover causally change CEO compensation levels and pay dispersion within European firms?*

**Research Motivation:** *By focusing on leadership changes observed in BoardEx, we can estimate how compensation responds to a governance shock without hand‑collecting policy data. This helps investors and regulators understand whether turnover acts as a disciplining mechanism on executive pay and internal pay gaps.*

### **2.2 Datasets and Key Variables**

#### **2.2.1 Sources**

1. **BoardEx - Individual Profile Employment**

    Includes data about Individual Profile Employment for executives in the BoardEx (Europe) universe.

    https://wrds-www.wharton.upenn.edu/pages/get-data/boardex/boardex-europe/individual-profile/individual-profile-employment/

2. **BoardEx - Individual Profile Details**

    Includes data about Individual Profile Details for executives in the BoardEx (Europe) universe.

    https://wrds-www.wharton.upenn.edu/pages/get-data/boardex/boardex-europe/individual-profile/individual-profile-details/

3. **Company Profile Details - BoardEx (Wharton Data Research Services)**

    Company Profile Details includes data such as location, market cap, and sector.
    
    https://wrds-www.wharton.upenn.edu/pages/get-data/boardex/boardex-europe/company-profile/company-profile-details/

4. **Fundamentals Annual - Compustat Global (Wharton Data Research Services)**

    Provides fundamental annual company information
    
    https://wrds-www.wharton.upenn.edu/pages/get-data/compustat-capital-iq-standard-poors/compustat/global-daily/fundamentals-annual/

5. **Annual Remuneration - BoardEx (Wharton Data Research Services):**
    
    Data such as salary, bonus, and other cash compensation
    
    https://wrds-www.wharton.upenn.edu/pages/get-data/boardex/boardex-europe/compensation-analysis/annual-remuneration/

6. **Firms in Stoxx 600 Index**

    STOXX 600 is a major stock index representing the performance of 600 large-, mid-, and small-capitalization companies across 17 developed European countries.

    https://www.stoxx.com/selection-lists

**Unit of observation:** *Executive–year within a firm (linked to firm-year fundamentals and leadership changes).*

#### **2.2.2 Key Variables**

| Variable | Type | Role (feature / target / instrument / ...) | Description |
|----------|------|---------------------------------------------|-------------|
| Total CEO compensation | Numeric | **Target** | Total annual compensation (salary + bonus + other cash/stock, in EUR) |
| Pay dispersion | Numeric | **Target / Outcome** | CEO pay relative to top-executive team (e.g., CEO-to-top-5 ratio) |
| CEO turnover indicator | Binary | **Treatment** | Flag for CEO change in a given year (from BoardEx role start/end) |
| Post‑turnover period | Binary | Feature | Indicator for years after turnover (event window) |
| Firm size (log assets / market cap) | Numeric | Feature | Scale and visibility of firm |
| Profitability (ROA / EBIT margin) | Numeric | Feature | Performance controls |
| Leverage | Numeric | Feature | Capital structure |
| Industry & country fixed effects | Categorical | Feature | Sector and institutional context |
| Executive tenure | Numeric | Feature | Human capital and bargaining power |

**Potential data quality issues:**  
- **Missing compensation components:** use multiple imputation or restrict to firms with complete pay breakdowns; report sensitivity to this choice.
- **Turnover date ambiguity:** define CEO change using role start/end dates and validate with overlapping roles; conduct robustness with alternative windows.
- **Reporting bias / top-coding:** winsorize extreme pay values; compare distributions by country to detect systematic reporting differences.
- **Selection bias in BoardEx coverage:** include a Stoxx 600 filter and check representativeness vs. population benchmarks.
- **Timing misalignment:** align fiscal-year fundamentals with compensation year; drop or lag inconsistent observations.
- **Currency and inflation effects:** convert to EUR and deflate using CPI to ensure comparability across years.

#### **2.2.3 Dataset Preview**

**1. Import pre installed WRDS package**

In [2]:
import wrds

import numpy as np
import pandas as pd
from pathlib import Path


**2. Establish connection with WRDS server**

- log in using your WRDS username and password

- set up a pgpass file to store the info

In [3]:
conn = wrds.Connection()

WRDS recommends setting up a .pgpass file.
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


**3. List all libraries**

- "Library" refers to databases on WRDS: e.g. CRSP, Compustat

- list_libraries() function to explore all subscribed databases

In [4]:
conn.list_libraries()

['aha_sample',
 'ahasamp',
 'auditsmp',
 'auditsmp_all',
 'bank',
 'bank_all',
 'bank_premium_samp',
 'banksamp',
 'block',
 'block_all',
 'boardex',
 'boardex_eur',
 'boardex_na',
 'boardex_trial',
 'boardex_uk',
 'boardsmp',
 'bvd_amadeus_trial',
 'bvd_bvdbankf_trial',
 'bvd_orbis_trial',
 'bvdsamp',
 'calcbench_trial',
 'calcbnch',
 'candid_samp',
 'cboe',
 'cboe_all',
 'cboe_sample',
 'cboesamp',
 'cddsamp',
 'ciq',
 'ciq_common',
 'ciq_keydev',
 'ciq_transactions',
 'ciqsamp',
 'ciqsamp_capstrct',
 'ciqsamp_common',
 'ciqsamp_keydev',
 'ciqsamp_pplintel',
 'ciqsamp_ratings',
 'ciqsamp_transactions',
 'ciqsamp_transcripts',
 'cisdmsmp',
 'columnar',
 'comp',
 'comp_bank_daily',
 'comp_execucomp',
 'comp_global_daily',
 'comp_na_daily_all',
 'comp_segments_hist_daily',
 'comp_snapshot',
 'compsamp',
 'compsamp_all',
 'compsamp_computext',
 'compsamp_snapshot',
 'compseg',
 'compsnap',
 'contrib',
 'contrib_as_filed_financials',
 'contrib_ceo_turnover',
 'contrib_corporate_culture',


In [5]:
df_db = pd.DataFrame(conn.list_libraries(), columns=['subbed_db']).sort_values(by='subbed_db', ascending=True)
df_db

,subbed_db
0,aha_sample
1,ahasamp
2,auditsmp
3,auditsmp_all
4,bank
...,...
204,wrdsapps_windices
205,wrdsappssamp_all
206,wrdssec_midas
207,zacksamp


In [6]:
with pd.option_context("display.max_rows", None):
    print(df_db)

                       subbed_db
0                     aha_sample
1                        ahasamp
2                       auditsmp
3                   auditsmp_all
4                           bank
5                       bank_all
6              bank_premium_samp
7                       banksamp
8                          block
9                      block_all
10                       boardex
11                   boardex_eur
12                    boardex_na
13                 boardex_trial
14                    boardex_uk
15                      boardsmp
16             bvd_amadeus_trial
17            bvd_bvdbankf_trial
18               bvd_orbis_trial
19                       bvdsamp
20               calcbench_trial
21                      calcbnch
22                   candid_samp
23                          cboe
24                      cboe_all
25                   cboe_sample
26                      cboesamp
27                       cddsamp
28                           ciq
29        

**4. List all datasets within a given library**

- databases contain many sub datasets

- list_tables() function to list all datasets

- specify which "library/database"

In [7]:
conn.list_tables(library='boardex_eur')

['eur_board_characteristics',
 'eur_board_dir_announcements',
 'eur_board_dir_committees',
 'eur_board_education_assoc',
 'eur_board_listed_assoc',
 'eur_board_nfp_assoc',
 'eur_board_other_assoc',
 'eur_board_unlisted_assoc',
 'eur_company_profile_advisors',
 'eur_company_profile_details',
 'eur_company_profile_market_cap',
 'eur_company_profile_sr_mgrs',
 'eur_company_profile_stocks',
 'eur_dir_characteristics',
 'eur_dir_education_assoc',
 'eur_dir_listed_assoc',
 'eur_dir_nfp_assoc',
 'eur_dir_other_assoc',
 'eur_dir_profile_achievements',
 'eur_dir_profile_details',
 'eur_dir_profile_education',
 'eur_dir_profile_emp',
 'eur_dir_profile_other_activ',
 'eur_dir_standard_remun',
 'eur_dir_unlisted_assoc',
 'eur_lookupcompany',
 'eur_lookuproles',
 'eur_lookupsalutations',
 'eur_ltip_compensation',
 'eur_ltip_wealth',
 'eur_options_compensation',
 'eur_options_wealth',
 'eur_wrds_company_dir_names',
 'eur_wrds_company_names',
 'eur_wrds_company_networks',
 'eur_wrds_company_profile',

**5. Query Data from WRDS Server**

- get_table() method

- straightforward if getting data from one single dataset

- specify which library/database and table/dataset to "get"

- can slice data by:
    - number of rows

    - column names

In [8]:
# Extract first 5 obs from comp.company

company = conn.get_table(library='comp', table='company', obs=5)
company

,conm,gvkey,add1,add2,add3,add4,addzip,busdesc,cik,city,...,sic,spcindcd,spcseccd,spcsrc,state,stko,weburl,dldte,ipodate,curr_sp500_flag
0,A & E PLASTIK PAK INC,001000,<NA>,<NA>,<NA>,<NA>,<NA>,A & E Plastik Pak Inc. is a commodity chemical...,<NA>,<NA>,...,3089,325,978,<NA>,<NA>,0,<NA>,1978-06-30,<NA>,0.0
1,A & M FOOD SERVICES INC,001001,1924 South Utica Avenue,<NA>,<NA>,<NA>,94104,<NA>,0000723576,Tulsa,...,5812,420,978,<NA>,OK,0,<NA>,1986-07-31,<NA>,0.0
2,AAI CORP,001002,124 Industry Lane,<NA>,<NA>,<NA>,21030-0126,"Textron Systems Corporation designs, develops,...",0001306124,Hunt Valley,...,3825,230,940,<NA>,MD,0,www.textronsystems.com,1977-03-31,<NA>,0.0
3,A.A. IMPORTING CO INC,001003,7700 Hall Street,<NA>,<NA>,<NA>,63125,"A.A. Importing Company, Inc. designs, manufact...",0000730052,St. Louis,...,5712,449,976,<NA>,MO,3,www.aaimporting.com,1992-04-30,<NA>,0.0
4,AAR CORP,001004,"One AAR Place, 1100 North Wood Dale Road",<NA>,<NA>,<NA>,60191,AAR Corp. provides products and services to co...,0000001750,Wood Dale,...,5080,110,925,B,IL,0,www.aarcorp.com,<NA>,1972-04-24,0.0


In [9]:
# eur_wrds_dir_profile_emp

df_emp = conn.get_table(library='boardex_eur', table='eur_wrds_dir_profile_emp', obs = 10)
df_emp

,rowtype,directorname,companyname,datestartrole,dateendrole,brdposition,rolename,fulltextdescription,ned,leadershipteam,primarykeyid,directorid,companyid,datestartroleflag,dateendroleflag,hocountryname,sector,orgtype,isin
0,Unlisted Organisations,Killko Caballero,US Dry Cleaning Corp (First Virtual Communicat...,2001-06-19,2002-10-28,Inside,President/CEO,<NA>,No,No,1731622.0,482055.0,9.0,10.0,15.0,United States,Consumer Services,Private,<NA>
1,Listed Organisations,William Burgess,FIRST ACTIVE PLC (De-listed 01/2004),1900-01-01,9999-12-31,Yes,NED,<NA>,Yes,No,1312708.0,328039.0,10.0,75.0,80.0,Republic Of Ireland,<NA>,Quoted,<NA>
2,Listed Organisations,Doctor António Palma Ramalho,FIRST ACTIVE PLC (De-listed 01/2004),1900-01-01,9999-12-31,Yes,Director - SD,<NA>,Yes,No,5617768.0,885377.0,10.0,75.0,80.0,Republic Of Ireland,<NA>,Quoted,<NA>
3,Listed Organisations,Clodagh NicCanna,FIRST ACTIVE PLC (De-listed 01/2004),1900-01-01,9999-12-31,No,Executive,<NA>,No,No,16213573.0,2765991.0,10.0,75.0,80.0,Republic Of Ireland,<NA>,Quoted,<NA>
4,Listed Organisations,Dick Milliken,FIRST ACTIVE PLC (De-listed 01/2004),1998-11-11,2004-01-28,Yes,NED,<NA>,Yes,No,2104000.0,512347.0,10.0,10.0,15.0,Republic Of Ireland,<NA>,Quoted,<NA>
5,Listed Organisations,John Callaghan,FIRST ACTIVE PLC (De-listed 01/2004),1999-04-01,2004-12-31,Yes,Chairman,<NA>,Yes,No,1146627.0,2763.0,10.0,20.0,25.0,Republic Of Ireland,<NA>,Quoted,<NA>
6,Listed Organisations,John Callaghan,FIRST ACTIVE PLC (De-listed 01/2004),1993-01-02,1997-12-31,Yes,NED,<NA>,Yes,No,1146623.0,2763.0,10.0,30.0,25.0,Republic Of Ireland,<NA>,Quoted,<NA>
7,Listed Organisations,John Callaghan,FIRST ACTIVE PLC (De-listed 01/2004),1997-01-02,1999-04-28,Yes,Vice Chairman,<NA>,Yes,No,1146624.0,2763.0,10.0,30.0,15.0,Republic Of Ireland,<NA>,Quoted,<NA>
8,Unlisted Organisations,Captain Aaron Bresnahan,First Agate Capital Corp,1993-06-01,1999-03-28,No,Various Positions,Maritime,No,No,6718065.0,880720.0,11.0,20.0,15.0,United States,Speciality & Other Finance,Private,<NA>
9,Unlisted Organisations,Adam Ebrahim,First Agate Capital Corp,1900-01-01,9999-12-31,Yes,Director - SD,<NA>,Yes,No,7328579.0,1261572.0,11.0,75.0,80.0,United States,Speciality & Other Finance,Private,<NA>


---

In [10]:
import numpy as np
import pandas as pd
from pathlib import Path

In [11]:
DATA_DIR = Path("..") / ".." / "data" / "data_project"

In [12]:
DATA_DIR = Path("../data")

df_emp = pd.read_csv(DATA_DIR / "executives_employment.csv")
df_emp.head(3)

,isin,datestartrole,directorname,companyname,dateendrole,brdposition,rolename,fulltextdescription,ned,primarykeyid,directorid,companyid,hocountryname
0,GB00B1YW4409,1900-01-01,John Yetman,3I GROUP PLC,9999-12-31,No,Executive,Investment Executive,No,1654504,8428,294,United Kingdom - England
1,GB00B1YW4409,2005-07-06,Rod Perry,3I GROUP PLC,9999-12-31,No,Consultant,Also Head of International Advisory Board of V...,No,2003633,11777,294,United Kingdom - England
2,GB00B1YW4409,1900-01-01,John De Zulueta Greenebaum,3I GROUP PLC,9999-12-31,No,Advisor,NaN,No,2542008,6474,294,United Kingdom - England


**1. Individual Profile Employment**

In [13]:
# Individual Profile Employment
df_emp = pd.read_csv(DATA_DIR / "executives_employment.csv")
df_emp.head(3)

,isin,datestartrole,directorname,companyname,dateendrole,brdposition,rolename,fulltextdescription,ned,primarykeyid,directorid,companyid,hocountryname
0,GB00B1YW4409,1900-01-01,John Yetman,3I GROUP PLC,9999-12-31,No,Executive,Investment Executive,No,1654504,8428,294,United Kingdom - England
1,GB00B1YW4409,2005-07-06,Rod Perry,3I GROUP PLC,9999-12-31,No,Consultant,Also Head of International Advisory Board of V...,No,2003633,11777,294,United Kingdom - England
2,GB00B1YW4409,1900-01-01,John De Zulueta Greenebaum,3I GROUP PLC,9999-12-31,No,Advisor,NaN,No,2542008,6474,294,United Kingdom - England


**2. Individual Profile Details**

In [14]:
# Individual Profile Details
df_exec_old = pd.read_csv(DATA_DIR / "executives_profile.csv")
df_exec_old.head()

/var/folders/yj/b9f3mv7j0bq585tjqs9_xgb00000gn/T/ipykernel_39384/2155953239.py:2: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  df_exec_old = pd.read_csv(DATA_DIR / "executives_profile.csv")


,directorid,directorname,title,forename1,forename2,forename3,forename4,usualname,surname,suffixtitle,...,dod,age,gender,recreations,directorvisible,dobflag,dodflag,wealthxid,primaryroleid,networksize
0,16,David Shaw,Mr,David,Evans,NaN,NaN,NaN,Shaw,MBA,...,9999-12-31,74.0,M,NaN,Yes,10,55,112260.0,1083314.0,5675.0
1,36,Kevin Kelly,Mr,Kevin,John,NaN,NaN,NaN,Kelly,FCA FCIB,...,2012-01-04,70.0,M,NaN,Yes,10,10,2247015.0,7117.0,NaN
2,37,Klaus Zwickel,Mr,Klaus,NaN,NaN,NaN,NaN,Zwickel,NaN,...,9999-12-31,86.0,M,NaN,Yes,10,55,NaN,3.0,468.0
3,39,Doctor Christopher Albrecht,Doctor,Christopher,J,C,NaN,NaN,Albrecht,PhD,...,9999-12-31,87.0,M,NaN,Yes,10,55,NaN,50962.0,78.0
4,42,Umberto Agnelli,Mr,Umberto,NaN,NaN,NaN,NaN,Agnelli,NaN,...,2004-05-27,69.0,M,NaN,Yes,10,10,NaN,133504.0,NaN


**3. Company Profile Details**

In [15]:
# Company Profile Details
df_firm = pd.read_csv(DATA_DIR / "company_profile.csv")
df_firm.head()

/var/folders/yj/b9f3mv7j0bq585tjqs9_xgb00000gn/T/ipykernel_39384/1818324475.py:2: DtypeWarning: Columns (9,10,14,15,16,17,18,19,20,21,29) have mixed types. Specify dtype option on import or set low_memory=False.
  df_firm = pd.read_csv(DATA_DIR / "company_profile.csv")


,isin,boardname,boardnameshort,hoaddress1,hoaddress2,hoaddress3,hoaddress4,hoaddress5,hocountryname,hotelnumber,...,successorcompanyid,ultimateparentcompanyid,boardid,ticker,countryofquote,primarystock,currency,mktcapitalisation,noemployees,revenue
0,NaN,1955 INVERSIONES SIMCAV SA,1955 INVERSIONES SIMCAV SA,NaN,NaN,NaN,NaN,NaN,Spain,NaN,...,NaN,NaN,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,FIRST ACTIVE PLC (De-listed 01/2004),FIRST ACTIVE PLC,NaN,NaN,NaN,NaN,NaN,Republic Of Ireland,NaN,...,783081.0,NaN,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,FIRST EAGLE FUND NV (De-listed 10/2011),FIRST EAGLE FUND NV,NaN,NaN,NaN,NaN,NaN,Netherlands Antilles,NaN,...,NaN,NaN,88,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,2M INVEST A/S,2M INVEST A/S,NaN,NaN,NaN,NaN,NaN,Denmark,NaN,...,NaN,NaN,264,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,GB0004618236,THREADNEEDLE UK SELECT TRUST LTD (UK Select Tr...,THREADNEEDLE UK SELECT TRUST LTD,Ground Floor Dorey Court,Admiral Park,St Peter Port,NaN,GY1 2HT,Guernsey,+44 (0)1 481 727 111,...,NaN,NaN,296,UKT,ENGLAND AND WALES,Yes,NaN,NaN,NaN,NaN


In [16]:
display(df_firm.describe())
display(df_firm.describe(include="object"))

,companypolicy,cikcode,previouscompanyid,successorcompanyid,boardid,mktcapitalisation,noemployees,revenue
count,0.0,1.041000e+03,6.901000e+03,6.593000e+03,4.450040e+05,4770.000000,4.383000e+03,4489.00000
mean,NaN,1.426868e+06,2.082375e+06,2.493534e+06,2.397661e+06,6238.678826,1.278777e+04,4566.55714
std,NaN,3.652228e+05,1.226734e+06,1.085201e+06,1.074114e+06,29254.949981,6.514137e+04,17911.67812
min,NaN,1.970000e+03,1.000000e+01,5.540000e+02,8.000000e+00,0.000000,1.000000e+00,0.00000
25%,NaN,1.167379e+06,1.141672e+06,1.703653e+06,1.673212e+06,78.000000,1.585000e+02,46.00000
50%,NaN,1.475011e+06,2.213311e+06,2.652171e+06,2.567914e+06,308.500000,1.015000e+03,336.00000
75%,NaN,1.665584e+06,3.182669e+06,3.426112e+06,3.275926e+06,1771.500000,6.105000e+03,2116.00000
max,NaN,2.079106e+06,4.100544e+06,4.101470e+06,4.101635e+06,563017.000000,2.476748e+06,336131.00000


,isin,boardname,boardnameshort,hoaddress1,hoaddress2,hoaddress3,hoaddress4,hoaddress5,hocountryname,hotelnumber,...,ccfaxnumber,sector,index,orgvisible,orgtype,ultimateparentcompanyid,ticker,countryofquote,primarystock,currency
count,6824,445004,445004,16905,12348,3593,14882,16462,445004,2698,...,2187,172966,1244,445004,445004,41559,6399,6823,6824,6408
unique,6677,444245,443604,14108,6404,1783,516,7563,55,2294,...,1907,52,89,2,10,10857,5914,50,2,1
top,LU0569974404,NB DISTRESSED DEBT INVESTMENT FUND LTD,NB DISTRESSED DEBT INVESTMENT FUND LTD,1 Royal Plaza,Amsterdam,Saint Peter Port,Paris,75008,Germany,+44 (0) 4 8175 0800,...,+46 (0) 8 735 57 44,Business Services,CDAX,No,Private,24776,TEL,FRANCE,Yes,USD
freq,4,12,12,28,243,139,1396,310,64496,12,...,7,21510,235,436587,389333,173,5,936,6230,6408


**4. Annual Renumeration**

In [17]:
# Annual Renumeration
df_renum_old = pd.read_csv(DATA_DIR / "annual_renumeration.csv")
df_renum_old.head()

,boardid,annualreportdate,rowtype,boardname,ned,directorname,rolename,currency,rolestatus,remchgelast,...,ltipvalue,intrvaloptaward,estvaloptaward,toteqatrisk,totremperiod,bonusratio,eqlinkremratio,wealthdelta,totaldirectcomp,perftotal
0,296,2016-12-01,SD Average,THREADNEEDLE UK SELECT TRUST LTD (UK Select Tr...,Yes,NaN,NaN,USD,NaN,0.14,...,NaN,NaN,NaN,NaN,35.0,NaN,NaN,2.0,35.0,NaN
1,296,2017-06-01,SD Average,THREADNEEDLE UK SELECT TRUST LTD (UK Select Tr...,Yes,NaN,NaN,USD,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,296,2017-06-01,Board Average,THREADNEEDLE UK SELECT TRUST LTD (UK Select Tr...,No,NaN,NaN,USD,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,296,2016-12-01,Board Average,THREADNEEDLE UK SELECT TRUST LTD (UK Select Tr...,No,NaN,NaN,USD,NaN,0.14,...,NaN,NaN,NaN,NaN,35.0,NaN,NaN,2.0,35.0,NaN
4,296,2017-06-01,SD Total,THREADNEEDLE UK SELECT TRUST LTD (UK Select Tr...,Yes,NaN,NaN,USD,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


**5. Firms in Stoxx 600 Index**

In [18]:
# Firms in EuroStoxx 600 Index
df_sxxp = pd.read_csv(DATA_DIR / "stoxx600_clean.csv", sep=";")
df_sxxp.head()

,Creation_Date,Internal_Key,ISIN,RIC,Instrument_Name,Country,Currency,Exchange,Index Membership,Rank (FINAL)
0,20260501,546078,NL0010273215,ASML.AS,ASML HLDG,NL,EUR,Euronext Amsterdam,Large,1
1,20260501,40054,GB0005405286,HSBA.L,HSBC,GB,GBP,London SE,Large,2
2,20260501,98952,GB0009895292,AZN.L,ASTRAZENECA,GB,GBP,London SE,Large,3
3,20260501,474577,CH0012032048,ROPC.S,ROCHE PS,CH,CHF,Six Swiss Exchange,Large,4
4,20260501,477408,CH0012005267,NOVN.S,NOVARTIS,CH,CHF,Six Swiss Exchange,Large,5


In [19]:
display(df_sxxp.describe())
display(df_sxxp.describe(include="object"))

,Creation_Date,Rank (FINAL)
count,600.0,600.000000
mean,20260501.0,300.500000
std,0.0,173.349358
min,20260501.0,1.000000
25%,20260501.0,150.750000
50%,20260501.0,300.500000
75%,20260501.0,450.250000
max,20260501.0,600.000000


,Internal_Key,ISIN,RIC,Instrument_Name,Country,Currency,Exchange,Index Membership
count,600,600,600,600,600,600,600,573
unique,600,600,600,600,17,8,16,3
top,546078,NL0010273215,ASML.AS,ASML HLDG,GB,EUR,London SE,Large
freq,1,1,1,1,128,297,127,200


---

---

### **2.3 Analytic Approach**

#### **2.3.1 Causal Inference**

1. Causal Graph

2. Backdoor Adjustment

**Justification:** We will model CEO turnover as a governance shock in a DAG and use backdoor adjustment to control for firm fundamentals that affect both turnover and pay. We will estimate the causal effect of turnover using pre/post (event‑study style) comparisons around the turnover year.

#### **2.3.2 Supervised Learning**

1. Linear / Ridge / Lasso Regression

2. Logistic Regression

3. Decision Tree / Random Forest

4. Gradient Boosting (XGBoost / LightGBM / sklearn GBM)

**Justification:** Supervised models will benchmark expected compensation conditional on firm and executive characteristics. Linear models provide interpretable baselines and covariate effects, while tree-based and boosting models capture nonlinearities/interactions for more accurate counterfactual pay predictions. We will use Optuna to tune boosting hyperparameters (e.g., depth, learning rate, subsampling) to avoid overfitting and compare against simpler baselines.

#### **2.3.3 Unsupervised Learning**

1. K-Means Clustering

2. Variational Autoencoder

**Justification:** Clustering will segment firms into comparable peer groups before causal estimation and highlight heterogeneous effects across turnover regimes. A VAE will learn low-dimensional representations of firm/executive profiles to detect anomalous pay structures and support exploratory subgroup analysis.

### **2.4 Evaluation Metric**

RMSE

**Justification: ...**

### **2.5 Research Work Plan**

| Step | Owner | Description |
|------|-------|-------------|
| 1 | Achmad | Data collection & merging from BoardEx/Compustat; define CEO turnover events |
| 2 | Kajetan | Data cleaning, missing-value strategy, currency/inflation adjustments |
| 3 | Achmad | EDA + descriptive stats; define pay dispersion metrics |
| 4 | Kajetan | Causal inference block (DAG, event-window design around turnover, ATE estimation) |
| 5 | Achmad | Supervised learning benchmark models + heterogeneity analysis |
| 6 | Kajetan | Unsupervised/generative clustering/embeddings; peer-group analysis |
| 7 | Achmad + Kajetan | Synthesis, robustness checks, and final write-up |

---

## **3. Data Understanding (Exploratory Data Analysis)**

### **3.1 Data Inspection and Descriptive Statistics**

**1. Individual Profile Employment**

In [20]:
df_emp.head(2)

,isin,datestartrole,directorname,companyname,dateendrole,brdposition,rolename,fulltextdescription,ned,primarykeyid,directorid,companyid,hocountryname
0,GB00B1YW4409,1900-01-01,John Yetman,3I GROUP PLC,9999-12-31,No,Executive,Investment Executive,No,1654504,8428,294,United Kingdom - England
1,GB00B1YW4409,2005-07-06,Rod Perry,3I GROUP PLC,9999-12-31,No,Consultant,Also Head of International Advisory Board of V...,No,2003633,11777,294,United Kingdom - England


| Column | Description |
|---|---|
| `isin` | International Securities Identification Number identifying the listed company/security. |
| `datestartrole` | Date when the individual started the reported role. |
| `directorname` | Name of the executive, director, or board member. |
| `companyname` | Name of the company where the individual holds or held the role. |
| `dateendrole` | Date when the role ended. Placeholder dates such as `9000-01-01` or `9999-12-31` indicate an ongoing or open-ended role. |
| `brdposition` | Indicates whether the role is a board position. Values are typically `Yes` or `No`. |
| `rolename` | Official title or position held by the individual, such as Group COO, Deputy Chair, or Independent Board Member. |
| `fulltextdescription` | Additional description or context about the role, if available. |
| `ned` | Indicates whether the individual is a non-executive director. Values are typically `Yes` or `No`. |
| `primarykeyid` | Unique identifier for the specific role/employment record. |
| `directorid` | Unique identifier for the individual executive or director. |
| `companyid` | Unique identifier for the company in the database. |
| `hocountryname` | Country of the company’s headquarters. |

In [21]:
display(df_emp.describe())
display(df_emp.describe(include="object"))

,primarykeyid,directorid,companyid
count,9.519700e+04,9.519700e+04,9.519700e+04
mean,1.086665e+07,1.605869e+06,3.494934e+05
std,4.892264e+06,8.766853e+05,7.973995e+05
min,2.190000e+02,3.600000e+01,2.940000e+02
25%,6.600024e+06,9.880770e+05,1.062700e+04
50%,1.142107e+07,1.600838e+06,2.435000e+04
75%,1.515448e+07,2.354550e+06,3.284700e+04
max,1.859297e+07,3.308284e+06,4.052924e+06


,isin,datestartrole,directorname,companyname,dateendrole,brdposition,rolename,fulltextdescription,ned,hocountryname
count,95197,95197,95197,95197,95197,95197,95197,55987,95197,95197
unique,569,5885,50631,587,4073,2,9222,44480,2,26
top,DE0005140008,1900-01-01,Doctor Roland Busch,DEUTSCHE BANK AG,9000-01-01,No,Executive,Also Member of the Executive Committee,No,France
freq,1481,8931,19,1481,19320,74617,5439,754,78780,17224


**Insights:**

-

-


**2. Individual Profile Details**

**3. Company Profile Details**

In [22]:
df_firm.head(2)

,isin,boardname,boardnameshort,hoaddress1,hoaddress2,hoaddress3,hoaddress4,hoaddress5,hocountryname,hotelnumber,...,successorcompanyid,ultimateparentcompanyid,boardid,ticker,countryofquote,primarystock,currency,mktcapitalisation,noemployees,revenue
0,NaN,1955 INVERSIONES SIMCAV SA,1955 INVERSIONES SIMCAV SA,NaN,NaN,NaN,NaN,NaN,Spain,NaN,...,NaN,NaN,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,FIRST ACTIVE PLC (De-listed 01/2004),FIRST ACTIVE PLC,NaN,NaN,NaN,NaN,NaN,Republic Of Ireland,NaN,...,783081.0,NaN,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN


**4. Annual Renumeration**

**5. Firms in Stoxx 600 Index**

In [23]:
df_sxxp

,Creation_Date,Internal_Key,ISIN,RIC,Instrument_Name,Country,Currency,Exchange,Index Membership,Rank (FINAL)
0,20260501,546078,NL0010273215,ASML.AS,ASML HLDG,NL,EUR,Euronext Amsterdam,Large,1
1,20260501,40054,GB0005405286,HSBA.L,HSBC,GB,GBP,London SE,Large,2
2,20260501,98952,GB0009895292,AZN.L,ASTRAZENECA,GB,GBP,London SE,Large,3
3,20260501,474577,CH0012032048,ROPC.S,ROCHE PS,CH,CHF,Six Swiss Exchange,Large,4
4,20260501,477408,CH0012005267,NOVN.S,NOVARTIS,CH,CHF,Six Swiss Exchange,Large,5
...,...,...,...,...,...,...,...,...,...,...
595,20260501,466514,FI0009002422,OUT1V.HE,OUTOKUMPU,FI,EUR,NASDAQ Helsinki,NaN,596
596,20260501,BE11G7,BE0974400328,AZE.BR,AZELIS GROUP,BE,EUR,Euronext Brussels,NaN,597
597,20260501,B1FW50,GB00B1FW5029,HOCM.L,HOCHSCHILD MINING,GB,GBP,London SE,Small,598
598,20260501,GB20F8,GB0006928617,UTG.L,UNITE GROUP PLC,GB,GBP,London SE,Small,599


___

**DF_REMUN FROM COMPANY_ID**

**Obtain Stoxx600 ID from company profile details**

In [ ]:
conn.get_data(
    library="boardex",
    table="eur_company_profile_details"
)

Approximately 444958 rows in boardex.eur_company_profile_details.


,name,nullable,type,comment
0,boardname,True,VARCHAR(128),Full company name or abbreviation in the case ...
1,boardnameshort,True,VARCHAR(255),Full company name only
2,hoaddress1,True,VARCHAR(128),None
3,hoaddress2,True,VARCHAR(128),None
4,hoaddress3,True,VARCHAR(128),None
5,hoaddress4,True,VARCHAR(64),None
6,hoaddress5,True,VARCHAR(64),None
7,hocountryname,True,VARCHAR(255),None
8,hotelnumber,True,VARCHAR(50),None
9,hofaxnumber,True,VARCHAR(50),None


In [36]:
isin_list = df_sxxp["ISIN"].dropna().unique().tolist()
isin_list

['NL0010273215',
 'GB0005405286',
 'GB0009895292',
 'CH0012032048',
 'CH0012005267',
 'CH0038863350',
 'GB00BP6MXD84',
 'DE0007236101',
 'FR0000120271',
 'FR0000121972',
 'ES0113900J37',
 'DE0007164600',
 'DE0008404005',
 'DE000ENER6Y0',
 'CH0012221716',
 'ES0144580Y14',
 'DK0060534915',
 'CH0244767585',
 'GB00B63H8491',
 'FR0000121014',
 'GB0007980591',
 'GB0002875804',
 'GB00B10RZP78',
 'ES0113211835',
 'FR0000120073',
 'NL0000235190',
 'FR0000073272',
 'IT0005239360',
 'DE0005557508',
 'FR0000131104',
 'GB0007188757',
 'GB00BN7SWP63',
 'FR0000120578',
 'IT0000072618',
 'FR0000120321',
 'CH0210483332',
 'CH0011075394',
 'GB00BDR05C01',
 'IT0003128367',
 'FR0000125486',
 'DE0006231004',
 'NL0011821202',
 'FR0000120628',
 'GB0002634946',
 'GB0031348658',
 'GB0008706128',
 'DE0008430026',
 'BE0974293251',
 'SE0015811963',
 'DE0007030009',
 'JE00B4T3BW64',
 'FI0009000681',
 'FR0000052292',
 'GB00B2B0DG97',
 'FR0000121667',
 'ES0148396007',
 'GB00BM8PJY71',
 'GB00B0SWJX34',
 'FI4000297767

---

**Obtain Related Data based on on ISIN**

**1. Individual Profile Employment**

In [ ]:
isin_list = df_sxxp["ISIN"].dropna().unique().tolist()

chunks = []
chunk_size = 1000

for i in range(0, len(isin_list), chunk_size):
    isin_chunk = tuple(isin_list[i:i + chunk_size])

    temp = conn.raw_sql("""
        SELECT *
        FROM boardex.eur_wrds_dir_profile_emp
        WHERE isin IN %(isins)s
        ORDER BY companyname, rolename, datestartrole, directorname
    """, params={"isins": isin_chunk})

    chunks.append(temp)

df_emp = pd.concat(chunks, ignore_index=True)

In [24]:
isin_list = df_sxxp["ISIN"].dropna().unique().tolist()

chunks = []
chunk_size = 1000

for i in range(0, len(isin_list), chunk_size):
    isin_chunk = tuple(isin_list[i:i + chunk_size])

    temp = conn.raw_sql("""
        SELECT *
        FROM boardex.eur_wrds_dir_profile_emp
        WHERE isin IN %(isins)s
        ORDER BY companyname, rolename, datestartrole, directorname
    """, params={"isins": isin_chunk})

    chunks.append(temp)

df_emp = pd.concat(chunks, ignore_index=True)

In [25]:
df_emp.head()

,rowtype,directorname,companyname,datestartrole,dateendrole,brdposition,rolename,fulltextdescription,ned,leadershipteam,primarykeyid,directorid,companyid,datestartroleflag,dateendroleflag,hocountryname,sector,orgtype,isin
0,Listed Organisations,Manoj Jain,3I GROUP PLC,2011-03-01,2014-05-28,No,Accountant,Fund Accountant,No,No,15144705.0,2614052.0,294.0,20.0,15.0,United Kingdom - England,Private Equity,Quoted,GB00B1YW4409
1,Listed Organisations,Pascal Lebard,3I GROUP PLC,1988-01-02,1989-12-31,No,Accounting Manager,<NA>,No,No,1332399.0,30680.0,294.0,30.0,25.0,United Kingdom - England,Private Equity,Quoted,GB00B1YW4409
2,Listed Organisations,Frédéric Vern,3I GROUP PLC,1900-01-01,9999-12-31,No,Advisor,<NA>,No,No,10093877.0,1915872.0,294.0,75.0,80.0,United Kingdom - England,Private Equity,Quoted,GB00B1YW4409
3,Listed Organisations,Kimmo Korpela,3I GROUP PLC,1900-01-01,9999-12-31,No,Advisor,Also Associate Director of 3i Sweden,No,No,5279095.0,1192043.0,294.0,75.0,80.0,United Kingdom - England,Private Equity,Quoted,GB00B1YW4409
4,Listed Organisations,Marcel van Poecke,3I GROUP PLC,1900-01-01,9999-12-31,No,Advisor,Benelux,No,No,2630695.0,330763.0,294.0,75.0,80.0,United Kingdom - England,Private Equity,Quoted,GB00B1YW4409


568

**2. Individual Profile Details**

In [26]:
directorid_list = df_emp["directorid"].dropna().unique().tolist()

chunks = []
chunk_size = 1000

for i in range(0, len(directorid_list), chunk_size):
    directorid_chunk = tuple(directorid_list[i:i + chunk_size])

    temp = conn.raw_sql("""
        SELECT directorname, title, forename1, surname, gender, directorid, primaryroleid
        FROM boardex.eur_dir_profile_details
        WHERE directorid IN %(directorids)s
        ORDER BY directorname ASC
    """, params={"directorids": directorid_chunk})

    chunks.append(temp)

df_exec = pd.concat(chunks, ignore_index=True)

In [27]:
df_exec.head(3)

,directorname,title,forename1,surname,gender,directorid,primaryroleid
0,Aaron Church,Mr,Aaron,Church,M,1348931.0,13060661.0
1,Admiral Rene Van Der Bruggen,Admiral,René,van der Bruggen,M,327778.0,4641608.0
2,Adriano Bandera,Mr,Adriano,Bandera,M,602849.0,4378308.0


In [28]:
df_exec_emp = df_exec.merge(df_emp, how = 'inner')[['directorid', 'directorname', 'title', 'companyname', 'companyid', 'rolename', 'datestartrole', 'dateendrole']]
df_exec_emp

,directorid,directorname,title,companyname,companyid,rolename,datestartrole,dateendrole
0,1348931.0,Aaron Church,Mr,3I GROUP PLC,294.0,Director - Infrastructure,2013-07-01,2022-04-28
1,1348931.0,Aaron Church,Mr,3I GROUP PLC,294.0,Partner,2022-04-01,9000-01-01
2,327778.0,Admiral Rene Van Der Bruggen,Admiral,AALBERTS NV (Aalberts Industries NV prior to 0...,384.0,Independent Board Member,2011-04-21,2014-04-22
3,602849.0,Adriano Bandera,Mr,A2A SPA,953.0,Director - SD,2008-02-22,2012-05-29
4,1520966.0,Adrian Yurkwich,Mr,3I GROUP PLC,294.0,Investment Executive,1995-03-01,1998-10-28
...,...,...,...,...,...,...,...,...
141817,601419.0,Yngve Bärgård,Mr,WARTSILA OYJ ABP,33085.0,Vice President - Supply,2005-08-01,2013-12-28
141818,1055598.0,Yves Bonte,Mr,YARA INTERNATIONAL ASA,500428.0,Executive VP,2010-01-01,2019-01-01
141819,1055598.0,Yves Bonte,Mr,YARA INTERNATIONAL ASA,500428.0,Executive VP - Business,2019-01-01,2020-02-28
141820,1541272.0,Yvonne Jamal,Ms,ZALANDO SE,2154485.0,Alternate Employee Representative,2015-06-02,2016-12-28


In [29]:
# --- 1.2  Identify CEO roles within STOXX 600 firms ---
CEO_PATTERN = r"chief executive|(?<!\w)ceo(?!\w)|managing director"
df_exec_emp["is_ceo"] = df_exec_emp["rolename"].str.lower().str.contains(CEO_PATTERN, na=False, regex=True)
df_ceo_emp = df_exec_emp[df_exec_emp["is_ceo"]].copy()

df_ceo_emp = df_ceo_emp.sort_values(by=['companyname', 'rolename', 'datestartrole'])

print(f"CEO role-records in STOXX 600 firms: {len(df_ceo_emp)}")
df_ceo_emp["rolename"].value_counts().head(10)

CEO role-records in STOXX 600 firms: 8044


rolename
Division CEO                       2690
CEO                                 727
Regional CEO                        599
President/CEO                       435
Chairman/CEO                        415
Division Chairman/Division CEO      256
Deputy CEO                          218
Division President/Division CEO     179
Division President/CEO              145
Division Deputy CEO                 124
Name: count, dtype: Int64

In [30]:
df_ceo_emp.head(3)

,directorid,directorname,title,companyname,companyid,rolename,datestartrole,dateendrole,is_ceo
2276,342366.0,Simon Borrows,Mr,3I GROUP PLC,294.0,CEO,2012-05-17,9000-01-01,True
1535,1297838.0,Maite Ballester Fornés,Ms,3I GROUP PLC,294.0,Regional CEO,2008-01-01,2014-03-28,True
15513,534736.0,Søren Skou,Mr,A.P. MOLLER-MAERSK A/S,250471.0,CEO,2016-07-01,2023-01-01,True


In [31]:
df_ceo_only = df_ceo_emp[df_ceo_emp["rolename"] == "CEO"]
df_ceo_only

,directorid,directorname,title,companyname,companyid,rolename,datestartrole,dateendrole,is_ceo
2276,342366.0,Simon Borrows,Mr,3I GROUP PLC,294.0,CEO,2012-05-17,9000-01-01,True
15513,534736.0,Søren Skou,Mr,A.P. MOLLER-MAERSK A/S,250471.0,CEO,2016-07-01,2023-01-01,True
15693,1526460.0,Vincent Clerc,Mr,A.P. MOLLER-MAERSK A/S,250471.0,CEO,2023-01-01,9000-01-01,True
1514,1587233.0,Luca Camerano,Mr,A2A SPA,953.0,CEO,2014-06-16,2017-07-11,True
2560,654106.0,Wim Pelsma,Mr,AALBERTS NV (Aalberts Industries NV prior to 0...,384.0,CEO,2012-04-26,2023-09-07,True
...,...,...,...,...,...,...,...,...,...
114595,8797.0,Peter Eckert,Mr,ZURICH INSURANCE GROUP AG,34200.0,CEO,2000-10-01,2001-07-28,True
141624,8796.0,Rolf Hüppi,Mr,ZURICH INSURANCE GROUP AG,34200.0,CEO,2002-05-16,2002-05-17,True
126772,31847.0,Martin Senn,Mr,ZURICH INSURANCE GROUP AG,34200.0,CEO,2010-01-01,2012-06-04,True
126773,31847.0,Martin Senn,Mr,ZURICH INSURANCE GROUP AG,34200.0,CEO,2012-09-01,2015-12-01,True


**4. Individual Annual Remuneration**

In [32]:
directorid_list = df_exec["directorid"].dropna().astype(int).unique().tolist()

chunks = []
chunk_size = 1000

for i in range(0, len(directorid_list), chunk_size):
    directorid_chunk = tuple(directorid_list[i:i + chunk_size])

    temp = conn.raw_sql("""
        SELECT 
            boardname,
            boardid,
            directorname,
            directorid,
            rolename,
            rolestatus,
            remchgelast,
            currency,
            salary,
            bonus,
            totalcompensation,
            annualreportdate
        FROM boardex.eur_dir_standard_remun
        WHERE directorid IN %(directorids)s
        ORDER BY boardname, directorname, annualreportdate ASC
    """, params={"directorids": directorid_chunk})

    chunks.append(temp)

df_remun = pd.concat(chunks, ignore_index=True) if chunks else pd.DataFrame()

In [39]:
df_remun_test = df_remun.sort_values(
    by=(['boardname', 'directorname', 'annualreportdate']),
    ascending=True
)

In [59]:
df_remun_test

,boardname,boardid,directorname,directorid,rolename,rolestatus,remchgelast,currency,salary,bonus,totalcompensation,annualreportdate
116460,029 GROUP SE (Mendarion SE prior to 08/2022),3593784.0,Leon Sander,2872572.0,Board Member - SD,Leon Sander has changed role on 01 Jul 2024,<NA>,USD,<NA>,<NA>,<NA>,2024-12-01
116461,029 GROUP SE (Mendarion SE prior to 08/2022),3593784.0,Leon Sander,2872572.0,MD,Leon Sander has changed role on 01 Jul 2024,<NA>,USD,<NA>,<NA>,<NA>,2024-12-01
116462,029 GROUP SE (Mendarion SE prior to 08/2022),3593784.0,Leon Sander,2872572.0,MD,Leon Sander joined this role on 01 Jul 2024,<NA>,USD,<NA>,<NA>,<NA>,<NA>
203335,1&1 AG (1&1 Drillisch AG prior to 06/2021),9638.0,Christine Schöneweis,2617542.0,Independent Board Member,Christine Schöneweis joined this role on 16 Ma...,<NA>,USD,<NA>,<NA>,<NA>,2023-12-01
203336,1&1 AG (1&1 Drillisch AG prior to 06/2021),9638.0,Christine Schöneweis,2617542.0,Independent Board Member,Christine Schöneweis joined this role on 16 Ma...,<NA>,USD,<NA>,<NA>,<NA>,2024-12-01
...,...,...,...,...,...,...,...,...,...,...,...,...
35023,Żabka Polska Sp Zoo,2228028.0,Tomasz Suchański,1851690.0,President/CEO,Tomasz Suchański joined this role on 01 Mar 2016,<NA>,USD,<NA>,<NA>,<NA>,2020-12-01
35024,Żabka Polska Sp Zoo,2228028.0,Tomasz Suchański,1851690.0,President/CEO,Tomasz Suchański joined this role on 01 Mar 2016,<NA>,USD,<NA>,<NA>,<NA>,2021-12-01
35025,Żabka Polska Sp Zoo,2228028.0,Tomasz Suchański,1851690.0,President/CEO,Tomasz Suchański joined this role on 01 Mar 2016,<NA>,USD,<NA>,<NA>,<NA>,2022-12-01
35026,Żabka Polska Sp Zoo,2228028.0,Tomasz Suchański,1851690.0,President/CEO,Tomasz Suchański joined this role on 01 Mar 2016,<NA>,USD,<NA>,<NA>,<NA>,2023-12-01


In [46]:
df_remun_test.annualreportdate.isna().sum()

np.int64(16657)

In [57]:
df_remun_test.totalcompensation.isna().sum()

np.int64(209303)

In [49]:
CEO_PATTERN = r"chief executive|(?<!\w)ceo(?!\w)|managing director"

In [54]:
df_remun_test_ceo = df_remun_test[df_remun_test["rolename"].str.lower().str.contains(CEO_PATTERN, na=False, regex=True)]
df_remun_test_ceo

,boardname,boardid,directorname,directorid,rolename,rolestatus,remchgelast,currency,salary,bonus,totalcompensation,annualreportdate
112054,1&1 AG (1&1 Drillisch AG prior to 06/2021),9638.0,Ralph Dommermuth,545942.0,CEO,Ralph Dommermuth joined this role on 01 Jan 2018,<NA>,USD,<NA>,<NA>,<NA>,2018-12-01
112055,1&1 AG (1&1 Drillisch AG prior to 06/2021),9638.0,Ralph Dommermuth,545942.0,CEO,Ralph Dommermuth joined this role on 01 Jan 2018,<NA>,USD,<NA>,<NA>,<NA>,2019-12-01
112056,1&1 AG (1&1 Drillisch AG prior to 06/2021),9638.0,Ralph Dommermuth,545942.0,CEO,Ralph Dommermuth joined this role on 01 Jan 2018,<NA>,USD,<NA>,<NA>,<NA>,2020-12-01
112057,1&1 AG (1&1 Drillisch AG prior to 06/2021),9638.0,Ralph Dommermuth,545942.0,CEO,Ralph Dommermuth joined this role on 01 Jan 2018,<NA>,USD,<NA>,<NA>,<NA>,2021-12-01
112058,1&1 AG (1&1 Drillisch AG prior to 06/2021),9638.0,Ralph Dommermuth,545942.0,CEO,Ralph Dommermuth joined this role on 01 Jan 2018,<NA>,USD,<NA>,<NA>,<NA>,2022-12-01
...,...,...,...,...,...,...,...,...,...,...,...,...
35022,Żabka Polska Sp Zoo,2228028.0,Tomasz Suchański,1851690.0,President/CEO,Tomasz Suchański joined this role on 01 Mar 2016,<NA>,USD,<NA>,<NA>,<NA>,2019-12-01
35023,Żabka Polska Sp Zoo,2228028.0,Tomasz Suchański,1851690.0,President/CEO,Tomasz Suchański joined this role on 01 Mar 2016,<NA>,USD,<NA>,<NA>,<NA>,2020-12-01
35024,Żabka Polska Sp Zoo,2228028.0,Tomasz Suchański,1851690.0,President/CEO,Tomasz Suchański joined this role on 01 Mar 2016,<NA>,USD,<NA>,<NA>,<NA>,2021-12-01
35025,Żabka Polska Sp Zoo,2228028.0,Tomasz Suchański,1851690.0,President/CEO,Tomasz Suchański joined this role on 01 Mar 2016,<NA>,USD,<NA>,<NA>,<NA>,2022-12-01


In [75]:
df_remun_new = df_remun_test[(df_remun_test["totalcompensation"] > 0) & df_remun_test['annualreportdate'].notnull()].drop_duplicates()
df_remun_new

,boardname,boardid,directorname,directorid,rolename,rolestatus,remchgelast,currency,salary,bonus,totalcompensation,annualreportdate
2,3I INFRASTRUCTURE PLC (3i Infrastructure Ltd p...,932863.0,Jenny Dunstan,1126372.0,NED,Jenny Dunstan joined this role on 20 Jul 2023,<NA>,USD,44.0,<NA>,44.0,2024-03-01
3,3I INFRASTRUCTURE PLC (3i Infrastructure Ltd p...,932863.0,Jenny Dunstan,1126372.0,NED,Jenny Dunstan joined this role on 20 Jul 2023,0.49,USD,67.0,<NA>,67.0,2025-03-01
120557,3I INFRASTRUCTURE PLC (3i Infrastructure Ltd p...,932863.0,Peter Wagner,2257.0,Independent NED,Peter Wagner joined this role on 13 Mar 2007,<NA>,USD,209.0,<NA>,209.0,2008-03-01
120558,3I INFRASTRUCTURE PLC (3i Infrastructure Ltd p...,932863.0,Peter Wagner,2257.0,Independent NED,Peter Wagner joined this role on 13 Mar 2007,-0.32,USD,103.0,<NA>,103.0,2009-03-01
196937,3I INFRASTRUCTURE PLC (3i Infrastructure Ltd p...,932863.0,Richard Laing,26240.0,Chairman (Independent NED),Richard Laing joined this role on 01 Jan 2016,<NA>,USD,50.0,<NA>,50.0,2016-03-01
...,...,...,...,...,...,...,...,...,...,...,...,...
7085,ZURICH INSURANCE GROUP AG,34200.0,Tom de Swaan,15011.0,Independent Chairman,Tom de Swaan has changed role on 11 Sep 2013,<NA>,USD,613.0,<NA>,613.0,2013-12-01
7086,ZURICH INSURANCE GROUP AG,34200.0,Tom de Swaan,15011.0,Independent Chairman,Tom de Swaan joined this role on 11 Sep 2013,0.75,USD,1011.0,<NA>,1011.0,2014-12-01
7088,ZURICH INSURANCE GROUP AG,34200.0,Tom de Swaan,15011.0,Chairman/Interim CEO,Tom de Swaan has changed role on 01 Dec 2015,<NA>,USD,738.0,<NA>,738.0,2015-12-01
7089,ZURICH INSURANCE GROUP AG,34200.0,Tom de Swaan,15011.0,Independent Chairman,Tom de Swaan has changed role on 07 Mar 2016,<NA>,USD,736.0,<NA>,736.0,2016-12-01


In [81]:
df_remun_new.boardname.unique()

<StringArray>
[                                                           '3I INFRASTRUCTURE PLC (3i Infrastructure Ltd prior to 07/2008)',
                                                                                'A&D PHARMA HOLDINGS NV (De-listed 03/2011)',
                                                                       'AARE-TESSIN AG FUR ELEKTRIZITAT (De-listed 06/2008)',
                                                                                                                   'ABB LTD',
                                             'ABERDEEN ASIAN INCOME FUND LTD (Abrdn Asian Income Fund Ltd prior to 06/2025)',
 'ABERDEEN FRONTIER MARKETS INVESTMENT COMPANY LTD (Advance Frontier Markets Fund Ltd prior to 04/2016) (De-listed 08/2020)',
                                                                                   'ABN AMRO HOLDING NV (De-listed 04/2008)',
       'ABRDN LATIN AMERICAN INCOME FUND LTD (Aberdeen Latin American Income Fund Ltd prior to 01/2022) 

In [77]:
df_remun_new.head()

,boardname,boardid,directorname,directorid,rolename,rolestatus,remchgelast,currency,salary,bonus,totalcompensation,annualreportdate
2,3I INFRASTRUCTURE PLC (3i Infrastructure Ltd p...,932863.0,Jenny Dunstan,1126372.0,NED,Jenny Dunstan joined this role on 20 Jul 2023,<NA>,USD,44.0,<NA>,44.0,2024-03-01
3,3I INFRASTRUCTURE PLC (3i Infrastructure Ltd p...,932863.0,Jenny Dunstan,1126372.0,NED,Jenny Dunstan joined this role on 20 Jul 2023,0.49,USD,67.0,<NA>,67.0,2025-03-01
120557,3I INFRASTRUCTURE PLC (3i Infrastructure Ltd p...,932863.0,Peter Wagner,2257.0,Independent NED,Peter Wagner joined this role on 13 Mar 2007,<NA>,USD,209.0,<NA>,209.0,2008-03-01
120558,3I INFRASTRUCTURE PLC (3i Infrastructure Ltd p...,932863.0,Peter Wagner,2257.0,Independent NED,Peter Wagner joined this role on 13 Mar 2007,-0.32,USD,103.0,<NA>,103.0,2009-03-01
196937,3I INFRASTRUCTURE PLC (3i Infrastructure Ltd p...,932863.0,Richard Laing,26240.0,Chairman (Independent NED),Richard Laing joined this role on 01 Jan 2016,<NA>,USD,50.0,<NA>,50.0,2016-03-01


---

**DF_REMUN FROM COMPANY_ID**

**Obtain Stoxx600 ID from company profile details**

In [26]:
conn.list_tables(library='boardex_eur')

['eur_board_characteristics',
 'eur_board_dir_announcements',
 'eur_board_dir_committees',
 'eur_board_education_assoc',
 'eur_board_listed_assoc',
 'eur_board_nfp_assoc',
 'eur_board_other_assoc',
 'eur_board_unlisted_assoc',
 'eur_company_profile_advisors',
 'eur_company_profile_details',
 'eur_company_profile_market_cap',
 'eur_company_profile_sr_mgrs',
 'eur_company_profile_stocks',
 'eur_dir_characteristics',
 'eur_dir_education_assoc',
 'eur_dir_listed_assoc',
 'eur_dir_nfp_assoc',
 'eur_dir_other_assoc',
 'eur_dir_profile_achievements',
 'eur_dir_profile_details',
 'eur_dir_profile_education',
 'eur_dir_profile_emp',
 'eur_dir_profile_other_activ',
 'eur_dir_standard_remun',
 'eur_dir_unlisted_assoc',
 'eur_lookupcompany',
 'eur_lookuproles',
 'eur_lookupsalutations',
 'eur_ltip_compensation',
 'eur_ltip_wealth',
 'eur_options_compensation',
 'eur_options_wealth',
 'eur_wrds_company_dir_names',
 'eur_wrds_company_names',
 'eur_wrds_company_networks',
 'eur_wrds_company_profile',

In [29]:
# Extract first 5 obs from comp.company

company = conn.get_table(library='boardex_eur', table='eur_company_profile_details', obs=5)
company.columns

Index(['boardname', 'boardnameshort', 'hoaddress1', 'hoaddress2', 'hoaddress3',
       'hoaddress4', 'hoaddress5', 'hocountryname', 'hotelnumber',
       'hofaxnumber', 'hourl', 'financialurl', 'companypolicy', 'ccaddress1',
       'ccaddress2', 'ccaddress3', 'ccaddress4', 'ccaddress5', 'cccountryname',
       'cctelnumber', 'ccfaxnumber', 'cikcode', 'sector', 'index',
       'orgvisible', 'orgtype', 'previouscompanyid', 'successorcompanyid',
       'ultimateparentcompanyid', 'boardid'],
      dtype='object')

In [ ]:
# Extract first 5 obs from comp.company

stock = conn.get_table(library='boardex_eur', table='eur_company_profile_stocks', obs=5)
stock.columns

Index(['boardname', 'ticker', 'isin', 'orgvisible', 'countryofquote',
       'primarystock', 'boardid'],
      dtype='object')

In [31]:
stock.head()

,boardname,ticker,isin,orgvisible,countryofquote,primarystock,boardid
0,THREADNEEDLE UK SELECT TRUST LTD (UK Select Tr...,UKT,GB0004618236,Yes,ENGLAND AND WALES,Yes,296.0
1,3U HOLDING AG (3U Telecom AG prior to 11/2007),UUU,DE0005167902,Yes,GERMANY,Yes,304.0
2,CURATIS HOLDING AG (Kinarus Therapeutics Holdi...,CURN,CH1330780979,Yes,SWITZERLAND,Yes,310.0
3,CURATIS HOLDING AG (Kinarus Therapeutics Holdi...,KNRS,CH0009115129,Yes,SWITZERLAND,No,310.0
4,ANOVO SA (A Novo prior to 06/2008) (De-listed ...,NOV,FR0010698217,Yes,FRANCE,Yes,352.0


In [38]:
isin_list = df_sxxp["ISIN"].dropna().unique().tolist()

chunks = []
chunk_size = 1000

for i in range(0, len(isin_list), chunk_size):
    isin_chunk = tuple(isin_list[i:i + chunk_size])

    temp = conn.raw_sql("""
        SELECT *
        FROM boardex_eur.eur_company_profile_stocks
        WHERE isin IN %(isins)s
    """, params={"isins": isin_chunk})

    chunks.append(temp)

df_comp = pd.concat(chunks, ignore_index=True)

In [39]:
df_comp.head()

,boardname,ticker,isin,orgvisible,countryofquote,primarystock,boardid
0,AALBERTS NV (Aalberts Industries NV prior to 0...,AALB,NL0000852564,Yes,NETHERLANDS,Yes,384.0
1,ABB LTD,ABB,CH0012221716,Yes,SWEDEN,No,422.0
2,ABB LTD,ABBN,CH0012221716,Yes,SWITZERLAND,Yes,422.0
3,ACCIONA SA,ANA,ES0125220311,Yes,SPAIN,Yes,595.0
4,ACCOR SA,AC,FR0000120404,Yes,FRANCE,Yes,598.0


In [40]:
len(df_comp)

491

In [ ]:
directorid_list = df_exec["directorid"].dropna().astype(int).unique().tolist()

chunks = []
chunk_size = 1000

for i in range(0, len(directorid_list), chunk_size):
    directorid_chunk = tuple(directorid_list[i:i + chunk_size])

    temp = conn.raw_sql("""
        SELECT 
            boardname,
            boardid,
            directorname,
            directorid,
            rolename,
            rolestatus,
            remchgelast,
            currency,
            salary,
            bonus,
            totalcompensation,
            annualreportdate
        FROM boardex.eur_dir_standard_remun
        WHERE directorid IN %(directorids)s
        ORDER BY boardname, directorname, annualreportdate ASC
    """, params={"directorids": directorid_chunk})

    chunks.append(temp)

df_remun = pd.concat(chunks, ignore_index=True) if chunks else pd.DataFrame()

---

In [70]:
df_final = df_remun_test_ceo[
    df_remun_test_ceo["totalcompensation"] > 0
].drop_duplicates()
df_final

,boardname,boardid,directorname,directorid,rolename,rolestatus,remchgelast,currency,salary,bonus,totalcompensation,annualreportdate
76548,A&D PHARMA HOLDINGS NV (De-listed 03/2011),1055962.0,Dragos Dinu,620369.0,CEO,Dragos Dinu joined this role on 24 Oct 2006,<NA>,USD,460.0,<NA>,460.0,2007-12-01
1026,ABB LTD,422.0,Doctor Ulrich Spiesshofer,486454.0,President/CEO,Doctor Ulrich Spiesshofer has changed role on ...,<NA>,USD,1236.0,1506.0,2742.0,2013-12-01
1027,ABB LTD,422.0,Doctor Ulrich Spiesshofer,486454.0,President/CEO,Doctor Ulrich Spiesshofer joined this role on ...,-0.01,USD,1618.0,2083.0,3701.0,2014-12-01
1028,ABB LTD,422.0,Doctor Ulrich Spiesshofer,486454.0,President/CEO,Doctor Ulrich Spiesshofer joined this role on ...,0.35,USD,1619.0,2573.0,4192.0,2015-12-01
1053,ABB LTD,422.0,Fred Kindle,4611.0,CEO Designate,Fred Kindle joined this role on 01 Sep 2004,<NA>,USD,351.0,<NA>,351.0,2004-12-01
...,...,...,...,...,...,...,...,...,...,...,...,...
276707,ZALANDO SE,2154485.0,Robert Gentz,1461707.0,Co-CEO,Robert Gentz joined this role in 2017,208.78,USD,103.0,<NA>,103.0,2023-12-01
276708,ZALANDO SE,2154485.0,Robert Gentz,1461707.0,Co-CEO,Robert Gentz joined this role in 2017,-0.86,USD,404.0,<NA>,404.0,2024-12-01
276716,ZALANDO SE,2154485.0,Rubin Ritter,1461711.0,Co-CEO,Rubin Ritter joined this role in 2017,<NA>,USD,73.0,<NA>,73.0,2019-12-01
276717,ZALANDO SE,2154485.0,Rubin Ritter,1461711.0,Co-CEO,Rubin Ritter joined this role in 2017,0.06,USD,79.0,<NA>,79.0,2020-12-01


In [73]:
df_final.head(50)

,boardname,boardid,directorname,directorid,rolename,rolestatus,remchgelast,currency,salary,bonus,totalcompensation,annualreportdate
76548,A&D PHARMA HOLDINGS NV (De-listed 03/2011),1055962.0,Dragos Dinu,620369.0,CEO,Dragos Dinu joined this role on 24 Oct 2006,<NA>,USD,460.0,<NA>,460.0,2007-12-01
1026,ABB LTD,422.0,Doctor Ulrich Spiesshofer,486454.0,President/CEO,Doctor Ulrich Spiesshofer has changed role on ...,<NA>,USD,1236.0,1506.0,2742.0,2013-12-01
1027,ABB LTD,422.0,Doctor Ulrich Spiesshofer,486454.0,President/CEO,Doctor Ulrich Spiesshofer joined this role on ...,-0.01,USD,1618.0,2083.0,3701.0,2014-12-01
1028,ABB LTD,422.0,Doctor Ulrich Spiesshofer,486454.0,President/CEO,Doctor Ulrich Spiesshofer joined this role on ...,0.35,USD,1619.0,2573.0,4192.0,2015-12-01
1053,ABB LTD,422.0,Fred Kindle,4611.0,CEO Designate,Fred Kindle joined this role on 01 Sep 2004,<NA>,USD,351.0,<NA>,351.0,2004-12-01
1054,ABB LTD,422.0,Fred Kindle,4611.0,President/CEO,Fred Kindle has changed role on 01 Jan 2005,<NA>,USD,993.0,306.0,1299.0,2005-12-01
1055,ABB LTD,422.0,Fred Kindle,4611.0,President/CEO,Fred Kindle joined this role on 01 Jan 2005,0.14,USD,1149.0,1540.0,2689.0,2006-12-01
1056,ABB LTD,422.0,Fred Kindle,4611.0,President/CEO,Fred Kindle joined this role on 01 Jan 2005,0.71,USD,1313.0,1746.0,3059.0,2007-12-01
1154,ABB LTD,422.0,Juergen Dormann,11152.0,Chairman/President/CEO,Juergen Dormann has changed role on 05 Sep 2002,<NA>,USD,1419.0,<NA>,1419.0,2002-12-01
1156,ABB LTD,422.0,Juergen Dormann,11152.0,Chairman/President/CEO,Juergen Dormann joined this role on 05 Sep 2002,0.58,USD,2414.0,<NA>,2414.0,2003-12-01


In [71]:
df_final.totalcompensation.isna().sum()

np.int64(0)

In [55]:
df_remun_test_ceo.annualreportdate.isna().sum()

np.int64(1082)

In [56]:
df_remun_test_ceo.totalcompensation.isna().sum()

np.int64(14639)

In [44]:
df_remun_test.rolename.value_counts().head(30)

rolename
Independent Board Member                                 46247
Board Member - SD                                        38418
Independent Director                                     32013
Employee Representative                                  16011
Director - SD                                            15612
Chairman                                                 11574
Board Member - ED                                         6469
Chairman/CEO                                              5504
Chairman (Independent Board Member)                       5228
Independent NED                                           5168
CEO                                                       4868
CFO                                                       4429
Shareholder Representative                                3965
Vice Chairman                                             3436
Independent Chairman                                      2715
Independent Shareholder Representative        

In [ ]:
df_ceo_only.head(20)

,directorid,directorname,title,companyname,companyid,rolename,datestartrole,dateendrole,is_ceo
2276,342366.0,Simon Borrows,Mr,3I GROUP PLC,294.0,CEO,2012-05-17,9000-01-01,True
15513,534736.0,Søren Skou,Mr,A.P. MOLLER-MAERSK A/S,250471.0,CEO,2016-07-01,2023-01-01,True
15693,1526460.0,Vincent Clerc,Mr,A.P. MOLLER-MAERSK A/S,250471.0,CEO,2023-01-01,9000-01-01,True
1514,1587233.0,Luca Camerano,Mr,A2A SPA,953.0,CEO,2014-06-16,2017-07-11,True
2560,654106.0,Wim Pelsma,Mr,AALBERTS NV (Aalberts Industries NV prior to 0...,384.0,CEO,2012-04-26,2023-09-07,True
2358,1542238.0,Stéphane Simonetta,Mr,AALBERTS NV (Aalberts Industries NV prior to 0...,384.0,CEO,2023-09-07,9000-01-01,True
3128,2124756.0,David Mindus,Mr,AB SAGAX,598834.0,CEO,2007-10-08,9000-01-01,True
231,32417.0,Björn Rosengren,Mr,ABB LTD,422.0,CEO,2020-03-01,2024-07-31,True
1839,2160278.0,Morten Wierod,Mr,ABB LTD,422.0,CEO,2024-08-01,9000-01-01,True
2089,1663966.0,Professor Doctor Hartmut Ehrlich,Professor Doctor,ABIVAX SA,2168332.0,CEO,2015-06-26,2023-05-05,True


In [ ]:
#test = df_ceo_only.merge(df_remun, on= 'directorid', how = 'inner')
# test.head(15)

In [ ]:
CEO_PATTERN = r"chief executive|(?<!\w)ceo(?!\w)|managing director"
df_remun["is_ceo"] = df_remun["rolename"].str.lower().str.contains(CEO_PATTERN, na=False, regex=True)
df_remun_new = df_remun[df_remun["is_ceo"]].copy()

In [ ]:
df_remun_new

,boardname,boardid,directorname,directorid,rolename,rolestatus,remchgelast,currency,salary,bonus,totalcompensation,annualreportdate,is_ceo
22,ADIDAS AG (Adidas-Salomon prior to 06/2006),792.0,Björn Gulden,1139597.0,CEO,Björn Gulden joined this role on 01 Jan 2023,<NA>,USD,2428.0,2023.0,4451.0,2023-12-01,True
23,ADIDAS AG (Adidas-Salomon prior to 06/2006),792.0,Björn Gulden,1139597.0,CEO,Björn Gulden joined this role on 01 Jan 2023,-0.26,USD,2278.0,2278.0,4556.0,2024-12-01,True
24,ADIDAS AG (Adidas-Salomon prior to 06/2006),792.0,Björn Gulden,1139597.0,CEO,Björn Gulden joined this role on 01 Jan 2023,0.05,USD,2584.0,2584.0,5168.0,2025-12-01,True
35,AIB GROUP PLC (Allied Irish Banks PLC prior to...,1505.0,Bernard Byrne,1072794.0,CEO,Bernard Byrne has changed role on 29 May 2015,<NA>,USD,524.0,<NA>,524.0,2015-12-01,True
36,AIB GROUP PLC (Allied Irish Banks PLC prior to...,1505.0,Bernard Byrne,1072794.0,CEO,Bernard Byrne joined this role on 29 May 2015,0.21,USD,527.0,<NA>,527.0,2016-12-01,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
10910,VOLKSWAGEN AG,32910.0,Thomas Schäfer,2364469.0,Division Chairman/Division CEO,Thomas Schäfer joined this role on 01 Jul 2022,-0.37,USD,1655.0,2199.0,3855.0,2023-12-01,True
10911,VOLKSWAGEN AG,32910.0,Thomas Schäfer,2364469.0,Division Chairman/Division CEO,Thomas Schäfer joined this role on 01 Jul 2022,-0.13,USD,1475.0,1777.0,3252.0,2024-12-01,True
10916,ZALANDO SE,2154485.0,Rubin Ritter,1461711.0,Co-CEO,Rubin Ritter joined this role in 2017,<NA>,USD,73.0,<NA>,73.0,2019-12-01,True
10917,ZALANDO SE,2154485.0,Rubin Ritter,1461711.0,Co-CEO,Rubin Ritter joined this role in 2017,0.06,USD,79.0,<NA>,79.0,2020-12-01,True


In [ ]:
df_remun_new.head(50)

,boardname,boardid,directorname,directorid,rolename,rolestatus,remchgelast,currency,salary,bonus,totalcompensation,annualreportdate,is_ceo
22,ADIDAS AG (Adidas-Salomon prior to 06/2006),792.0,Björn Gulden,1139597.0,CEO,Björn Gulden joined this role on 01 Jan 2023,<NA>,USD,2428.0,2023.0,4451.0,2023-12-01,True
23,ADIDAS AG (Adidas-Salomon prior to 06/2006),792.0,Björn Gulden,1139597.0,CEO,Björn Gulden joined this role on 01 Jan 2023,-0.26,USD,2278.0,2278.0,4556.0,2024-12-01,True
24,ADIDAS AG (Adidas-Salomon prior to 06/2006),792.0,Björn Gulden,1139597.0,CEO,Björn Gulden joined this role on 01 Jan 2023,0.05,USD,2584.0,2584.0,5168.0,2025-12-01,True
35,AIB GROUP PLC (Allied Irish Banks PLC prior to...,1505.0,Bernard Byrne,1072794.0,CEO,Bernard Byrne has changed role on 29 May 2015,<NA>,USD,524.0,<NA>,524.0,2015-12-01,True
36,AIB GROUP PLC (Allied Irish Banks PLC prior to...,1505.0,Bernard Byrne,1072794.0,CEO,Bernard Byrne joined this role on 29 May 2015,0.21,USD,527.0,<NA>,527.0,2016-12-01,True
37,AIB GROUP PLC (Allied Irish Banks PLC prior to...,1505.0,Bernard Byrne,1072794.0,CEO,Bernard Byrne joined this role on 29 May 2015,0.04,USD,600.0,<NA>,600.0,2017-12-01,True
38,AIB GROUP PLC (Allied Irish Banks PLC prior to...,1505.0,Bernard Byrne,1072794.0,CEO,Bernard Byrne joined this role on 29 May 2015,0.02,USD,575.0,<NA>,575.0,2018-12-01,True
52,AIR LIQUIDE SA,1168.0,Alain Joly,15840.0,Chairman/CEO,Alain Joly joined this role in May 1995,<NA>,USD,762.0,<NA>,762.0,2000-12-01,True
71,AIR LIQUIDE SA,1168.0,Benoît Potier,5736.0,Chairman/CEO,Benoît Potier has changed role on 10 May 2006,<NA>,USD,1244.0,1668.0,2912.0,2006-12-01,True
72,AIR LIQUIDE SA,1168.0,Benoît Potier,5736.0,Chairman/CEO,Benoît Potier has changed role on 10 May 2006,<NA>,USD,1244.0,1668.0,2912.0,2006-12-01,True


In [ ]:
len(df_remun_new.boardname.unique())

176

In [ ]:
df_ceo_emp[df_ceo_emp['directorid'] == 534736.0]

,directorid,directorname,title,companyname,companyid,rolename,datestartrole,dateendrole,is_ceo
15515,534736.0,Søren Skou,Mr,A.P. MOLLER-MAERSK A/S,250471.0,Executive VP/Division CEO,2003-11-01,2007-07-01,True
15516,534736.0,Søren Skou,Mr,A.P. MOLLER-MAERSK A/S,250471.0,Partner/Division CEO,2007-07-01,2011-12-28,True
15514,534736.0,Søren Skou,Mr,A.P. MOLLER-MAERSK A/S,250471.0,Division CEO,2012-01-16,2016-07-01,True
15513,534736.0,Søren Skou,Mr,A.P. MOLLER-MAERSK A/S,250471.0,CEO,2016-07-01,2023-01-01,True


In [ ]:
df_remun[df_remun['directorid'] == 534736.0]

,boardname,boardid,directorname,directorid,rolename,rolestatus,remchgelast,currency,salary,bonus,totalcompensation,annualreportdate
10459,NOKIA OYJ,22335.0,Søren Skou,534736.0,Independent Board Member,Søren Skou joined this role on 21 May 2019,<NA>,USD,179.0,<NA>,179.0,2019-12-01
10460,NOKIA OYJ,22335.0,Søren Skou,534736.0,Independent Board Member,Søren Skou joined this role on 21 May 2019,0.06,USD,196.0,<NA>,196.0,2020-12-01
10461,NOKIA OYJ,22335.0,Søren Skou,534736.0,Independent Board Member,Søren Skou joined this role on 21 May 2019,0.03,USD,199.0,<NA>,199.0,2021-12-01
10462,NOKIA OYJ,22335.0,Søren Skou,534736.0,Vice Chair (Independent Board Member),Søren Skou has changed role on 05 Apr 2022,<NA>,USD,225.0,<NA>,225.0,2022-12-01
10463,NOKIA OYJ,22335.0,Søren Skou,534736.0,Vice Chair (Independent Board Member),Søren Skou joined this role on 05 Apr 2022,0.05,USD,248.0,<NA>,248.0,2023-12-01
10464,NOKIA OYJ,22335.0,Søren Skou,534736.0,Vice Chair (Independent Board Member),Søren Skou joined this role on 05 Apr 2022,-0.07,USD,228.0,<NA>,228.0,2024-12-01


---

In [ ]:
display(df_ceo_emp[df_ceo_emp['directorid'] == 882159.0])
display(df_remun[df_remun['directorid'] == 882159.0])

,directorid,directorname,title,companyname,companyid,rolename,datestartrole,dateendrole,is_ceo
95550,882159.0,Mike Kerner,Mr,ZURICH INSURANCE GROUP AG,34200.0,Division Regional CEO,2009-06-01,2012-09-01,True
95549,882159.0,Mike Kerner,Mr,ZURICH INSURANCE GROUP AG,34200.0,Division CEO,2012-09-01,2015-10-01,True


,boardname,boardid,directorname,directorid,rolename,rolestatus,remchgelast,currency,salary,bonus,totalcompensation,annualreportdate
8759,MUNCHENER RUCKVERSICHERUNGS AG (Munich Re),21321.0,Mike Kerner,882159.0,Board Member - ED,Mike Kerner joined this role on 01 Jan 2023,<NA>,USD,1583.0,<NA>,1583.0,2023-12-01
8760,MUNCHENER RUCKVERSICHERUNGS AG (Munich Re),21321.0,Mike Kerner,882159.0,Board Member - ED,Mike Kerner joined this role on 01 Jan 2023,0.65,USD,1746.0,816.0,2561.0,2024-12-01


___

In [ ]:
import pandas as pd

# Make copies
remun = df_remun.copy()
ceo_emp = df_ceo_emp.copy()

# Convert date columns
remun["annualreportdate"] = pd.to_datetime(remun["annualreportdate"], errors="coerce")
ceo_emp["datestartrole"] = pd.to_datetime(ceo_emp["datestartrole"], errors="coerce")
ceo_emp["dateendrole"] = pd.to_datetime(ceo_emp["dateendrole"], errors="coerce")

# Optional: if dateendrole is missing, treat as still active
ceo_emp["dateendrole"] = ceo_emp["dateendrole"].fillna(pd.Timestamp.today())

# Keep only CEO rows from employment table
ceo_emp_only = ceo_emp[ceo_emp["is_ceo"] == True].copy()

# Merge remuneration with CEO role periods
df_remun_ceo = remun.merge(
    ceo_emp_only[[
        "directorid",
        "directorname",
        "companyid",
        "companyname",
        "rolename",
        "datestartrole",
        "dateendrole"
    ]],
    on="directorid",
    how="inner",
    suffixes=("_remun", "_ceo")
)

# Filter where annual report date is within CEO role period
df_remun_ceo = df_remun_ceo[['companyid', 'companyname', 'directorname_ceo', 'directorid', 'rolename_ceo', 'rolename_remun', 'currency', 'salary', 'bonus', 'totalcompensation', 'remchgelast', 'annualreportdate', 'datestartrole', 'dateendrole']]
df_remun_ceo = df_remun_ceo[df_remun_ceo['totalcompensation'] > 0]

df_remun_ceo_filtered = df_remun_ceo[
    (df_remun_ceo["annualreportdate"] >= df_remun_ceo["datestartrole"]) &
    (df_remun_ceo["annualreportdate"] <= df_remun_ceo["dateendrole"])
].copy()

In [ ]:
df_remun_ceo.sort_values(by=['companyname', 'datestartrole', 'annualreportdate']).tail(50)

,companyid,companyname,directorname_ceo,directorid,rolename_ceo,rolename_remun,currency,salary,bonus,totalcompensation,remchgelast,annualreportdate,datestartrole,dateendrole
2873,34200.0,ZURICH INSURANCE GROUP AG,Doctor Thomas Buberl,654634.0,Regional CEO,CEO,USD,1697.0,1720.0,3418.0,0.08,2022-12-01,2009-01-01,2012-04-28 00:00:00.000000
2878,34200.0,ZURICH INSURANCE GROUP AG,Doctor Thomas Buberl,654634.0,Regional CEO,CEO,USD,1821.0,1932.0,3753.0,0.23,2023-12-01,2009-01-01,2012-04-28 00:00:00.000000
2883,34200.0,ZURICH INSURANCE GROUP AG,Doctor Thomas Buberl,654634.0,Regional CEO,CEO,USD,1708.0,1956.0,3664.0,-0.02,2024-12-01,2009-01-01,2012-04-28 00:00:00.000000
2888,34200.0,ZURICH INSURANCE GROUP AG,Doctor Thomas Buberl,654634.0,Regional CEO,CEO,USD,1915.0,<NA>,1915.0,<NA>,NaT,2009-01-01,2012-04-28 00:00:00.000000
15109,34200.0,ZURICH INSURANCE GROUP AG,Mike Kerner,882159.0,Division Regional CEO,Board Member - ED,USD,1583.0,<NA>,1583.0,<NA>,2023-12-01,2009-06-01,2012-09-01 00:00:00.000000
15111,34200.0,ZURICH INSURANCE GROUP AG,Mike Kerner,882159.0,Division Regional CEO,Board Member - ED,USD,1746.0,816.0,2561.0,0.65,2024-12-01,2009-06-01,2012-09-01 00:00:00.000000
1642,34200.0,ZURICH INSURANCE GROUP AG,Claudia Dill,1017142.0,Division CEO,Independent Board Member,USD,138.0,<NA>,138.0,<NA>,2021-12-01,2009-09-01,2012-07-01 00:00:00.000000
15110,34200.0,ZURICH INSURANCE GROUP AG,Mike Kerner,882159.0,Division CEO,Board Member - ED,USD,1583.0,<NA>,1583.0,<NA>,2023-12-01,2012-09-01,2015-10-01 00:00:00.000000
15112,34200.0,ZURICH INSURANCE GROUP AG,Mike Kerner,882159.0,Division CEO,Board Member - ED,USD,1746.0,816.0,2561.0,0.65,2024-12-01,2012-09-01,2015-10-01 00:00:00.000000
1643,34200.0,ZURICH INSURANCE GROUP AG,Claudia Dill,1017142.0,Regional CEO,Independent Board Member,USD,138.0,<NA>,138.0,<NA>,2021-12-01,2015-05-01,2020-09-02 00:00:00.000000


---

In [ ]:
conn.list_libraries()

['aha_sample',
 'ahasamp',
 'auditsmp',
 'auditsmp_all',
 'bank',
 'bank_all',
 'bank_premium_samp',
 'banksamp',
 'block',
 'block_all',
 'boardex',
 'boardex_eur',
 'boardex_na',
 'boardex_trial',
 'boardex_uk',
 'boardsmp',
 'bvd_amadeus_trial',
 'bvd_bvdbankf_trial',
 'bvd_orbis_trial',
 'bvdsamp',
 'calcbench_trial',
 'calcbnch',
 'candid_samp',
 'cboe',
 'cboe_all',
 'cboe_sample',
 'cboesamp',
 'cddsamp',
 'ciq',
 'ciq_common',
 'ciq_keydev',
 'ciq_transactions',
 'ciqsamp',
 'ciqsamp_capstrct',
 'ciqsamp_common',
 'ciqsamp_keydev',
 'ciqsamp_pplintel',
 'ciqsamp_ratings',
 'ciqsamp_transactions',
 'ciqsamp_transcripts',
 'cisdmsmp',
 'columnar',
 'comp',
 'comp_bank_daily',
 'comp_execucomp',
 'comp_global_daily',
 'comp_na_daily_all',
 'comp_segments_hist_daily',
 'comp_snapshot',
 'compsamp',
 'compsamp_all',
 'compsamp_computext',
 'compsamp_snapshot',
 'compseg',
 'compsnap',
 'contrib',
 'contrib_as_filed_financials',
 'contrib_ceo_turnover',
 'contrib_corporate_culture',


In [ ]:
conn.list_tables(library='boardex_eur')

['eur_board_characteristics',
 'eur_board_dir_announcements',
 'eur_board_dir_committees',
 'eur_board_education_assoc',
 'eur_board_listed_assoc',
 'eur_board_nfp_assoc',
 'eur_board_other_assoc',
 'eur_board_unlisted_assoc',
 'eur_company_profile_advisors',
 'eur_company_profile_details',
 'eur_company_profile_market_cap',
 'eur_company_profile_sr_mgrs',
 'eur_company_profile_stocks',
 'eur_dir_characteristics',
 'eur_dir_education_assoc',
 'eur_dir_listed_assoc',
 'eur_dir_nfp_assoc',
 'eur_dir_other_assoc',
 'eur_dir_profile_achievements',
 'eur_dir_profile_details',
 'eur_dir_profile_education',
 'eur_dir_profile_emp',
 'eur_dir_profile_other_activ',
 'eur_dir_standard_remun',
 'eur_dir_unlisted_assoc',
 'eur_lookupcompany',
 'eur_lookuproles',
 'eur_lookupsalutations',
 'eur_ltip_compensation',
 'eur_ltip_wealth',
 'eur_options_compensation',
 'eur_options_wealth',
 'eur_wrds_company_dir_names',
 'eur_wrds_company_names',
 'eur_wrds_company_networks',
 'eur_wrds_company_profile',

### **3.2 Data Visualization**

---

## **4. Data Preprocessing**

### **4.1 Data Cleaning**

#### **4.1.1 Missing Values**

#### **4.1.2 Duplicated Values**

#### **4.1.3 Outliers**

### **4.2 Feature Selection**

---

## **5. Causal Inference**

---

## **6. Supervised Learning**

---

## **7. Unsupervised Learning**

---

## **8. Results and Conclusion**

---

---
## 7. Results *(complete for final submission)*


### 7a. Causal Inference

In [ ]:
# Causal inference analysis

### 7b. Supervised Learning

In [ ]:
# Supervised learning analysis

### 7c. Unsupervised / Generative

In [ ]:
# Unsupervised / generative analysis

## 8. Discussion & Conclusion *(complete for final submission)*

*Synthesise findings across all three method blocks. What does each lens reveal that the others miss? What are the limitations of your analysis?*
